# Convert the CSV Data to RDF

## Create Deep JSON

In [3]:
import pandas as pd

party_csv = pd.read_csv("../data/input/party.csv")
party_name_csv = pd.read_csv("../data/input/party_name.csv")
relationship_csv = pd.read_csv("../data/input/relationship.csv")
relationship_type_csv = pd.read_csv("../data/input/relationship_type.csv")
party_code_csv = pd.read_csv("../data/input/party_code.csv")
party_code_type_csv = pd.read_csv("../data/input/party_code_type.csv")


display(party_csv.head())

,party_id,identifier,color,party_name_id,relationship_id_1,relationship_id_2,social_group_id,party_category_id,election_result_id,period_of_service_id,...,voting_issue_party_party_id13,voting_issue_notes13,voting_issue_sources13,dissolution_date_precision,dissolution_date_datetime,dissolution_date_notes,dissolution_date_sources,translations_languages_code3,translations_description3,translations_hls_link_external_url3
0,1,svp-ch,#4B8A3E,[1],"[3,9]","[5,12,35,69,90,98,141,174,192,193,232,260,276,...",[1],[16],"[1,2,3,4,5,6,7,8,9,10,11,12,13,14,106,107,108,...","[1,2,3,4,5,6,7,8,9,562,563,564,565,566,567,568...",...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,sp-ch,#F0554D,[2],[],"[13,36,70,91,99,142,175,196,201,202,233,234,26...",[2],[1],"[15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,...","[10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,fdplib-ch,#3B79BF,[3],"[1,6]","[18,38,72,93,101,135,144,177,203,205,210,240,2...",[3],[15],"[43,44,45,46,148,149,150,151,6290,7226,7227,72...","[38,39,40,41]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,mitte-ch,#D6862B,[4],"[2,4]","[21,31,40,43,61,75,95,103,125,145,146,163,180,...",[4],[14],"[47,152,6284,7250,7251,7252,7253]","[42,43,611,612]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,gruene-ch,#84B547,"[6,7,8,9,199]",[],"[22,44,45,48,49,53,54,56,78,79,96,104,105,106,...",[5],[13],"[48,49,50,51,52,53,54,55,56,57,58,59,60,153,15...","[44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
def parse_list(s):
    if pd.isna(s) or s == "[]":
        return []
    return [int(item) for item in s.strip("[]").split(",")]

In [11]:
# the following structure should be created:

# parties = {
#    1: {
#         "names": [
#             {"name": "name1", "from_date": "date1", "to_date": "date2"},
#             {"name": "name2", "from_date": "date3", "to_date": "date4"},
#         ],
#         "relationships": [...],
#     },
#     2: {
#         "names": [...],
#         "relationships": [...],
#     },
#     ...
# ]

parties = {}

for _, row in party_csv.iterrows():
    
    party = {}
    
    party_id = row["party_id"]
    
    # get party names
    party_name_ids = parse_list(row["party_name_id"])

    names = []
    
    for party_name_id in party_name_ids:
        name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "translations_name1"].item()
        from_date = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "from_date_datetime"].item()
        to_date = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "to_date_datetime"].item()
        name_entry = {"id": party_name_id, "name": name, "from_date": from_date, "to_date": to_date}
        names.append(name_entry)

    
    # sort by from_date
    names.sort(key=lambda x: x["from_date"])
    party["names"] = names


    # get relationships, they are given in two columns, relationship_id_1 and relationship_id_2.
    # all child-of relationships go to `child_relationships`, all other relationships go to `relationships`.
    # maybe I'm missing out on some meaning of the two columns, but for now I will just combine them and sort by from_date.
    relationships_1 = []
    relationships_2 = []

    # get relationships_1
    relationship_ids_1 = parse_list(row["relationship_id_1"])

    for relationship_id in relationship_ids_1:
        relationship_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_2_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        date_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        date_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        relationship_entry = {"id": relationship_id, "type_label": relationship_label, "party": relationship_to, "from_date": date_from, "to_date": date_to}
        if relationship_entry["type_label"] != "child-of":
            relationships_1.append(relationship_entry)
        else:
            relationships_2.append(relationship_entry)

    # get relationships_2
    relationship_ids_2 = parse_list(row["relationship_id_2"])


    for relationship_id in relationship_ids_2:
        relationship_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_1_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        date_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        date_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        relationship_entry = {"id": relationship_id, "type_label": relationship_label, "party": relationship_from, "from_date": date_from, "to_date": date_to}
        if relationship_entry["type_label"] != "child-of":
            relationships_1.append(relationship_entry)
        else:
            relationships_2.append(relationship_entry)

    # sort relationships by from_date and add them to the party dictionary
    relationships_1.sort(key=lambda x: x["from_date"])
    relationships_2.sort(key=lambda x: x["from_date"])
    party["relationships"] = relationships_1
    party["child_relationships"] = relationships_2

    # different identifiers for the same party (e.g. bfs-code, wikidata, etc.)
    party_code_ids = parse_list(row["party_code_id"])
    party_codes = []

    for party_code_id in party_code_ids:
        code = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "code"].item()
        code_type = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "party_code_type_id"].item()
        code_type_label = party_code_type_csv.loc[party_code_type_csv["party_code_type_id"] == code_type, "identifier"].item()

        code_entry = {"id": party_code_id, "code": code, "type_label": code_type_label}
        party_codes.append(code_entry)

    party_codes.append({"code": row["identifier"], "type_label": "ap-identifier"})  # add the identifier from the party.csv row

    party["party_codes"] = party_codes


    # add party to parties dictionary (finish line operations)
    parties[party_id] = party

In [12]:
import json

print(json.dumps(parties, indent=2, ensure_ascii=False))

{
  "1": {
    "names": [
      {
        "id": 1,
        "name": "Schweizerische Volkspartei",
        "from_date": "1971-09-22T12:00:00",
        "to_date": NaN
      }
    ],
    "relationships": [
      {
        "id": 3,
        "type_label": "successor-of",
        "party": 11,
        "from_date": "1971-09-22T12:00:00",
        "to_date": NaN
      },
      {
        "id": 9,
        "type_label": "successor-of",
        "party": 18,
        "from_date": "1971-09-22T12:00:00",
        "to_date": NaN
      },
      {
        "id": 5,
        "type_label": "split-from",
        "party": 13,
        "from_date": "2008-11-01T12:00:00",
        "to_date": NaN
      }
    ],
    "child_relationships": [
      {
        "id": 12,
        "type_label": "child-of",
        "party": 20,
        "from_date": "1971-09-22T12:00:00",
        "to_date": NaN
      },
      {
        "id": 35,
        "type_label": "child-of",
        "party": 35,
        "from_date": "1971-09-22T12:00:00",
   

## Convert the Deep JSON to RDF

In [ ]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace

g = Graph()